5.1 定义超参数 + 读取数据

In [20]:
import torch
import torch.nn as nn
from torchvision import datasets ,transforms
from torch.utils.data import DataLoader
import torch.optim as optim
#1、超参数
batch_size = 256 
epochs = 10
lr = 0.01
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

transform= transforms.Compose([transforms.ToTensor(),transforms.Normalize(0.5,0.5)])

mnist_train = datasets.FashionMNIST(root='../data',train=True,download=True,transform=transform)
mnist_test = datasets.FashionMNIST(root='..data',train=False,download=True,transform=transform)

train_loader= DataLoader(mnist_train,batch_size=batch_size,shuffle=True)
test_loader = DataLoader(mnist_test,batch_size=batch_size,shuffle=True)


cpu


In [15]:
for X,y in train_loader:
    print(X.shape,y.shape,y)
    break


torch.Size([256, 1, 28, 28]) torch.Size([256]) tensor([0, 8, 3, 3, 9, 7, 2, 1, 8, 6, 3, 9, 8, 8, 0, 3, 4, 2, 0, 9, 3, 2, 0, 1,
        6, 7, 5, 5, 4, 9, 6, 7, 3, 4, 0, 5, 5, 3, 6, 2, 8, 7, 6, 7, 3, 0, 3, 7,
        3, 9, 8, 5, 7, 6, 5, 3, 5, 9, 4, 6, 4, 9, 3, 6, 8, 7, 5, 8, 3, 0, 0, 4,
        6, 6, 9, 1, 7, 6, 7, 1, 3, 5, 7, 6, 5, 2, 8, 6, 2, 2, 8, 7, 7, 5, 8, 7,
        5, 9, 2, 8, 3, 7, 2, 9, 5, 7, 4, 0, 8, 4, 2, 8, 7, 6, 8, 3, 1, 1, 2, 6,
        9, 6, 9, 9, 6, 6, 6, 1, 3, 8, 6, 2, 6, 1, 9, 8, 3, 9, 3, 1, 3, 6, 6, 8,
        1, 4, 3, 7, 0, 6, 7, 7, 2, 7, 5, 2, 5, 0, 7, 6, 6, 0, 2, 4, 7, 3, 3, 8,
        8, 1, 2, 6, 8, 5, 4, 2, 9, 7, 4, 8, 3, 1, 6, 4, 9, 1, 6, 5, 3, 4, 7, 3,
        6, 1, 2, 1, 6, 7, 7, 8, 8, 9, 7, 0, 3, 3, 3, 5, 8, 7, 5, 2, 8, 3, 6, 3,
        1, 5, 5, 8, 3, 2, 7, 5, 1, 8, 0, 4, 7, 2, 6, 0, 6, 5, 2, 8, 1, 1, 8, 7,
        2, 8, 1, 6, 2, 2, 0, 4, 9, 2, 2, 9, 9, 9, 3, 1])


5.2 定义模型

In [17]:
class FashionMLP(nn.Module):
    def __init__(self):
        super(FashionMLP,self).__init__()
        self.flatten = nn.Flatten() #将28*28展开成 784
        self.main = nn.Sequential(
            nn.Linear(784,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        x = self.flatten(x)
        logits = self.main(x)
        return logits
    
model = FashionMLP().to(device)

5.3 定义损失函数与优化器

In [21]:
criterion = nn.CrossEntropyLoss() 
optimizer = optim.SGD(model.parameters(),lr= lr)

5.4 训练循环

In [25]:
print(f"正在 {device} 上启动训练...")

for epoch in range(epochs):
    model.train()#设置为训练模式
    running_loss = 0
    for X,y in train_loader:
        X,y = X.to(device),y.to(device)
        y_hat = model(X)
        loss = criterion(y_hat,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()#设置为评估模式
    correct = 0
    total = 0 
    with torch.no_grad():#验证阶段不计算梯度
        for X,y in test_loader:
            X,y = X.to(device),y.to(device)
            y_hat = model(X)
            _,predicted = torch.max(y_hat,1)#返回概率最大的类别索引值
            total += len(y)
            correct += (predicted == y).sum().item()
    print(f"Epoch: {epoch+1}    Loss: {running_loss/len(train_loader):.4f}    Accuracy: {100*correct/total :.2f}%")

print("训练完成！")


正在 cpu 上启动训练...
Epoch: 1    Loss: 0.4465    Accuracy: 83.03%
Epoch: 2    Loss: 0.4406    Accuracy: 83.26%
Epoch: 3    Loss: 0.4340    Accuracy: 83.32%
Epoch: 4    Loss: 0.4291    Accuracy: 83.41%
Epoch: 5    Loss: 0.4242    Accuracy: 83.81%
Epoch: 6    Loss: 0.4194    Accuracy: 83.67%
Epoch: 7    Loss: 0.4152    Accuracy: 83.95%
Epoch: 8    Loss: 0.4106    Accuracy: 83.97%
Epoch: 9    Loss: 0.4068    Accuracy: 83.97%
Epoch: 10    Loss: 0.4032    Accuracy: 84.25%
训练完成！
